In [1]:
import os
import glob
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import json
from pathlib import Path
import numpy as np
from tqdm.notebook import tqdm

# collect scores

In [2]:
# Base directory containing all expression runs
base_dir = "../../runs/expression/qnorm"
# base_dir = "/runs/expression" # for mb machine

# Function to extract scores from TensorBoard event files
def extract_scores_from_events(event_file_path, run_name):
	"""Extract all scalar values at a specific step from TensorBoard event file."""
	try:
		ea = EventAccumulator(event_file_path)
		ea.Reload()
		
		scores = []
		for tag in ea.Tags()['scalars']:
			events = ea.Scalars(tag)
			# Find events at or closest to target step
			step_values = [(event.step, event.value) for event in events]
						
			for step, value in step_values:
				if step > 100: # skip test runs
					scores.append({"step": step, "value": value, "run_name": run_name, "metric": tag})
		return scores
	except Exception as e:
		print(f"Error processing {event_file_path}: {e}")
		return {}

# Main function to traverse all subdirectories and collect scores
def collect_all_scores(base_dir):
    """Traverse all subdirectories recursively and collect scores at specified step."""
    all_results = []
    all_dirs = list(os.walk(base_dir))
    for ind, (root, dirs, files) in enumerate(all_dirs):
        # Find TensorBoard event files in this directory
        event_files = glob.glob(os.path.join(root, "events.out.tfevents*"))
        if not event_files:
            continue
        print (ind, " of ",len(all_dirs))            
        # print(f"Processing: {root}")
        for event_file in event_files:
            # Only keep the part of the path after base_dir in run_name
            rel_run_name = os.path.relpath(root, base_dir)
            scores = extract_scores_from_events(event_file, run_name=rel_run_name)
            if scores:
                all_results.extend(scores)
                # print(f"  Found {len(scores)} metrics")
            else:
                pass
                # print(f"  No scores found")
    return all_results
	
# Execute the collection
print(f"Starting to collect scores from all subdirectories...")
results = collect_all_scores(base_dir)

print(f"\nCollected data from {len(results)} runs")

# Convert to DataFrame for easier analysis
if results:
	df = pd.DataFrame.from_records(results)
	print(f"\nDataFrame shape: {df.shape}")
	print("\nColumns:", df.columns.tolist())
	
	# Display first few rows
	print("\nFirst few results:")
	display(df.head())
	
	# Save to CSV
	output_file = f"expression_scores.csv"
	df.to_csv(output_file, index=False, compression="gzip")
	print(f"\nResults saved to {output_file}")
else:
	print("No results found!")

Starting to collect scores from all subdirectories...
2  of  212
6  of  212
10  of  212
15  of  212
19  of  212
23  of  212
28  of  212
30  of  212
32  of  212
34  of  212
38  of  212
39  of  212
42  of  212
46  of  212
49  of  212
53  of  212
57  of  212
61  of  212
65  of  212
68  of  212
72  of  212
76  of  212
81  of  212
86  of  212
91  of  212
97  of  212
101  of  212
104  of  212
109  of  212
111  of  212
115  of  212
117  of  212
126  of  212
129  of  212
138  of  212
151  of  212
156  of  212
162  of  212
163  of  212
164  of  212
177  of  212
190  of  212
195  of  212
196  of  212
197  of  212
203  of  212
207  of  212
208  of  212
211  of  212

Collected data from 1394564 runs

DataFrame shape: (1394564, 4)

Columns: ['step', 'value', 'run_name', 'metric']

First few results:


,step,value,run_name,metric
0,150,3.122101,qnorm_v1/20250710-153301,loss/iterations/train
1,200,2.902649,qnorm_v1/20250710-153301,loss/iterations/train
2,250,2.862545,qnorm_v1/20250710-153301,loss/iterations/train
3,300,2.611175,qnorm_v1/20250710-153301,loss/iterations/train
4,350,2.554961,qnorm_v1/20250710-153301,loss/iterations/train



Results saved to expression_scores.csv


In [4]:
df["run_name"].unique()

array(['qnorm_v1/20250710-153301',
       'qnorm_v1/resume_20250710-153301/model_80000.pth/20250714-172332',
       'qnorm_v1_atac_seq/20250627-214947',
       'qnorm_v1_atac_seq/resume_/20250627-214947/model_50000.pth/20250629-003205',
       'qnorm_v1_mouse_large_1seg_12_layers/20250802-013050',
       'qnorm_v1_mouse_triplet_loss/20250722-235145',
       'qnorm_v1_mouse_margin_loss/20250723-182033',
       'qnorm_v1_mouse_corr_loss/20250723-200828',
       'qnorm_v1_mouse_sci_bert/20250725-143959',
       'qnorm_v1_mouse_sci_bert/20250725-151659',
       'qnorm_v1_mouse_sci_bert_frozen_11_layer/20250725-200203',
       'qnorm_v1_without_sci_bert/20250725-185613',
       'qnorm_v1_mouse_large_1seg_lr/20250814-012249',
       'qnorm_v1_mouse_large_1seg_Tweedie/20250802-141258',
       'qnorm_v1_human_large_1seg_bw/20250810-024702',
       'qnorm_v1_mouse_large_1seg_mae/20250814-125324',
       'qnorm_v1_mouse_large_1seg/20250801-123959',
       'qnorm_v1_mouse_base_2seg/20250717-16264

# Output scores for selected run in table-ready format

In [5]:
df = pd.read_csv("expression_scores.csv", compression="gzip")
# df = pd.read_csv("expression_scores_25-07-2025.csv", compression="gzip")
df.head(2)

,step,value,run_name,metric
0,150,3.122101,qnorm_v1/20250710-153301,loss/iterations/train
1,200,2.902649,qnorm_v1/20250710-153301,loss/iterations/train


In [6]:
df["run_name"].unique()

array(['qnorm_v1/20250710-153301',
       'qnorm_v1/resume_20250710-153301/model_80000.pth/20250714-172332',
       'qnorm_v1_atac_seq/20250627-214947',
       'qnorm_v1_atac_seq/resume_/20250627-214947/model_50000.pth/20250629-003205',
       'qnorm_v1_mouse_large_1seg_12_layers/20250802-013050',
       'qnorm_v1_mouse_triplet_loss/20250722-235145',
       'qnorm_v1_mouse_margin_loss/20250723-182033',
       'qnorm_v1_mouse_corr_loss/20250723-200828',
       'qnorm_v1_mouse_sci_bert/20250725-143959',
       'qnorm_v1_mouse_sci_bert/20250725-151659',
       'qnorm_v1_mouse_sci_bert_frozen_11_layer/20250725-200203',
       'qnorm_v1_without_sci_bert/20250725-185613',
       'qnorm_v1_mouse_large_1seg_lr/20250814-012249',
       'qnorm_v1_mouse_large_1seg_Tweedie/20250802-141258',
       'qnorm_v1_human_large_1seg_bw/20250810-024702',
       'qnorm_v1_mouse_large_1seg_mae/20250814-125324',
       'qnorm_v1_mouse_large_1seg/20250801-123959',
       'qnorm_v1_mouse_base_2seg/20250717-16264

In [15]:
selected_run = 'qnorm_v1_mouse_large_1seg_quantloss_2попытка/20250814-234346'

In [16]:
sel = df.query("run_name == @selected_run")
_met = [i for i in sel.metric.unique() if ("valid" in i) and ("v1" in i or "loss" in i) and ("samples" in i) and not ("_4_" in i)]
sel = sel.query("metric in @_met").dropna(subset=["value"])
results = sel.groupby("metric").apply(lambda x: x.loc[x["value"].idxmin()] if x["metric"].str.contains("loss").any() else x.loc[x["value"].idxmax()])
results.reset_index(drop=True).sort_values("metric", ascending=False)

,step,value,run_name,metric
5,564480,0.526587,qnorm_v1_mouse_large_1seg_quantloss_2попытка/2...,score_predictions_Expression_dataset_v1_mm10_C...
4,544320,0.542597,qnorm_v1_mouse_large_1seg_quantloss_2попытка/2...,score_predictions_Expression_dataset_v1_mm10_C...
3,1751680,0.658740,qnorm_v1_mouse_large_1seg_quantloss_2попытка/2...,pearson_corr_genes_Expression_dataset_v1_mm10_...
2,1426880,0.189147,qnorm_v1_mouse_large_1seg_quantloss_2попытка/2...,pearson_corr_cells_Expression_dataset_v1_mm10_...
1,105280,0.169951,qnorm_v1_mouse_large_1seg_quantloss_2попытка/2...,loss_cls_loss/samples/valid
0,105280,0.169951,qnorm_v1_mouse_large_1seg_quantloss_2попытка/2...,loss/samples/valid


In [17]:
# select steps when scpre prediction on v1 is best (2 steps probably, one for human, one for mouse)
steps = results[results.metric.str.startswith('score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid [slow-10]')]["step"].values

In [18]:
for step in steps:
	# sel.query("metric in @_met and step == @step").reset_index(drop=True).sort_values("metric", ascending=False)
	_met_sorted = [
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid [slow-10]',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
]
	print (selected_run, end="\t")
	print (step, end="\t")
	for met in _met_sorted:
		# print (met, end="\t")
		v = sel.query("metric == @met and step == @step")["value"].values[0]
		print (v, end="\t")
	print("\n------")

qnorm_v1_mouse_large_1seg_quantloss_2попытка/20250814-234346	564480	0.5265874862670898	0.0881529226899147	0.5933788418769836	
------


In [32]:
for step in steps:
	# sel.query("metric in @_met and step == @step").reset_index(drop=True).sort_values("metric", ascending=False)
	_met_sorted = [
 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid [slow-10]' ,
 'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
]
	print (selected_run, end="\t")
	print (step, end="\t")
	for met in _met_sorted:
		# print (met, end="\t")
		v = sel.query("metric == @met and step == @step")["value"].values[0]
		print (v, end="\t")
	print("\n------")

qnorm_v1_human_large_1seg_quantloss/20250819-000423	617400	0.016463052481412888	0.12176636606454849	0.6644256114959717	
------


In [14]:
for step in steps:
	# sel.query("metric in @_met and step == @step").reset_index(drop=True).sort_values("metric", ascending=False)
	_met_sorted = [
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid [slow-10]',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
]
	print (selected_run, end="\t")
	print (step, end="\t")
	for met in _met_sorted:
		# print (met, end="\t")
		v = sel.query("metric == @met and step == @step")["value"].values[0]
		print (v, end="\t")
	print("\n------")

qnorm_v1_human_base_1seg/20250806-145550	421750	0.053571298718452454	

IndexError: index 0 is out of bounds for axis 0 with size 0

44800
0.1085683479905128	0.110475406050682	-0.0988152027130127	-0.0105760768055915	0.5893904566764832	0.5265839695930481	

In [62]:
_met

['loss/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid']

In [ ]:
import sys
sys.path.append("/home/jovyan/dnalm/downstream_tasks/annotation")

from transcript_GenePredictionDataset_letter_level_CADUSEUS_intergenic import AnnotationDataset
./data/tokenizers/t2t_1000h_multi_32k/
dataset = AnnotationDataset("/home/jovyan/shares/SR003.nfs2/chr_dataset_human/chr_dataset_gena_gpt_valid.hdf5", "/home/jovyan/shares/SR003.nfs2/chr_dataset_human/chr_dataset_gena_gpt_test.hdf5")

for i in range(len(dataset)):
    print(dataset[i])
    break

/workspace-SR003.nfs2/aspeedok/aspeedok/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: __init__() missing 2 required positional arguments: 'tokenizer' and 'tokenizer_caduseus'